In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    AutoModelForMaskedLM,
)

from chop import MaseGraph
import chop.passes as passes


# checkpoint = "bert-base-uncased"
# tokenizer_checkpoint = "bert-base-uncased"

# checkpoint = "roberta-base"
# tokenizer_checkpoint = "roberta-base"

checkpoint = "albert/albert-base-v2"
tokenizer_checkpoint = "albert/albert-base-v2"


dataset_name = "xu-song/cc100-samples"

dataset = load_dataset(dataset_name, "en", split="train[:100%]")
# # add dummy label
# dataset = dataset.map(lambda e: {"labels": 0, "text": e["text"]})
tokenizer = AutoTokenizer.from_pretrained(tokenizer_checkpoint)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
    )

# Tokenize
dataset = dataset.map(tokenize_function, batched=True)

# split the dataset in train and test
dataset = dataset.train_test_split(test_size=0.2)

print(dataset)
# print(dataset["train"][0])

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)



DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})


In [ ]:
model = AutoModelForMaskedLM.from_pretrained(checkpoint)
# print(model)
model.cuda()

mg = MaseGraph(
    model,
    hf_input_names=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)

mg, _ = passes.init_metadata_analysis_pass(mg)
# following line not working
# mg, _ = passes.add_common_metadata_analysis_pass(mg)

# model = mg.model
# print(mg.model)
training_args = TrainingArguments(
    output_dir = "mase-trainer",
    report_to="none",
    num_train_epochs=3,
)

trainer = Trainer(
    mg.model,
    training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)


eval_results = trainer.evaluate()
print(f"Evaluation loss: {eval_results['eval_loss']}")

trainer.train()

eval_results = trainer.evaluate()
print(f"Evaluation loss: {eval_results['eval_loss']}")

mg.export("test_1")


Some weights of the model checkpoint at albert/albert-base-v2 were not used when initializing AlbertForMaskedLM: ['albert.pooler.bias', 'albert.pooler.weight']
- This IS expected if you are initializing AlbertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing AlbertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/tmp/ipykernel_180128/355114300.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluation loss: 8.586492538452148


Step,Training Loss
500,3.148400
1000,3.029500
1500,2.703900
2000,2.643000
2500,2.425000


RuntimeError: 
            Some tensors share memory, this will lead to duplicate memory on disk and potential differences when loading them again: [{'predictions.decoder.weight', 'albert.embeddings.word_embeddings.weight'}].
            A potential way to correctly save your model is to use `save_model`.
            More information at https://huggingface.co/docs/safetensors/torch_shared_tensors
            

In [ ]:
mg2 = MaseGraph.from_checkpoint("test_1")


trainer = Trainer(
    mg2.model,
    training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)


eval_results = trainer.evaluate()
print(f"Evaluation loss: {eval_results['eval_loss']}")

INFO     Exporting MaseGraph to test_1.pt, test_1.mz
INFO     Exporting GraphModule to test_1.pt
INFO     Exporting MaseMetadata to test_1.mz
